# AI 기반 지반함몰 위험도 예측 실습

도시 지하 인프라 데이터를 이용하여 Low, Medium, High의 세 단계 위험도를 분류합니다.

실습 흐름: 데이터 확인 → Model A → Model B → Model C → 성능 비교 → 변수 중요도 해석

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
candidates = [
    Path("../data/hannam_sinkhole_data.xlsx"),
    Path("data/hannam_sinkhole_data.xlsx"),
    Path("hannam_sinkhole_data.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("data/hannam_sinkhole_data.xlsx 파일을 찾을 수 없습니다.")

df = pd.read_excel(DATA_PATH, sheet_name="training_data")
print("shape:", df.shape)
df.head()

## Step 1. 데이터 구조 확인

결측치와 중복, 위험도 분포를 확인합니다. 데이터 품질을 직접 복구하는 활동은 이번 60분 실습 범위에서 제외합니다.

In [ ]:
print(df.info())
print("결측치:", int(df.isna().sum().sum()))
print("중복 행:", int(df.duplicated().sum()))
display(df["risk"].value_counts().sort_index().rename(index={0:"Low", 1:"Medium", 2:"High"}))

In [ ]:
risk_counts = df["risk"].value_counts().sort_index()
ax = risk_counts.plot(kind="bar", color=["#4CAF50", "#FFC107", "#F44336"])
ax.set_title("Risk class distribution")
ax.set_xlabel("risk")
ax.set_ylabel("count")
plt.show()

In [ ]:
def evaluate_model(name, features):
    X = df[features].copy()
    y = df["risk"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    model = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "model_object": model,
        "features": features,
        "y_test": y_test,
        "pred": pred,
    }
    return result

## Step 2. Model A — 밀집도 변수 2개

TODO: 상수도관 밀집도와 하수도관 밀집도를 선택하고 기준 모델을 만드세요.

In [ ]:
water_features = [f"water_Y_{year}" for year in range(5, 60, 5)]
sewer_features = [f"sewer_Y_{year}" for year in range(5, 25, 5)]

basic_features = ["water_Density", "sewer_total_density_km_km2"]
raw_features = water_features + ["water_Density"] + sewer_features + [
    "sewer_total_density_km_km2", "cavity_count"
]
derived_features = [
    "total_water_length", "total_sewer_length", "pipe_total_density",
    "water_sewer_ratio", "old_pipe_ratio", "recent_pipe_ratio",
    "cavity_density_interaction",
]
engineered_features = raw_features + derived_features

# TODO: basic_features가 올바른지 확인한 뒤 실행하세요.
result_a = evaluate_model("Model A: density only", basic_features)
print("Accuracy:", round(result_a["accuracy"], 3))
print("Macro F1:", round(result_a["macro_f1"], 3))

### 생각해 볼 질문

- 밀집도 두 개만으로 세 등급을 충분히 구분할 수 있나요?
- Accuracy와 Macro F1-score가 다른 이유는 무엇인가요?

## Step 3. Model B — 원본 변수 전체

TODO: 시공 시기별 관로 길이, 밀집도, 공동 개수를 포함하여 모델을 학습하세요.

In [ ]:
# TODO: raw_features를 사용해 Model B를 학습하세요.
result_b = evaluate_model("Model B: raw features", raw_features)
print("Accuracy:", round(result_b["accuracy"], 3))
print("Macro F1:", round(result_b["macro_f1"], 3))

## Step 4. Model C — 파생 변수 포함

TODO: 원본 변수와 파생 변수를 함께 사용하여 모델을 학습하세요.

In [ ]:
# TODO: engineered_features를 사용해 Model C를 학습하세요.
result_c = evaluate_model("Model C: engineered features", engineered_features)
print("Accuracy:", round(result_c["accuracy"], 3))
print("Macro F1:", round(result_c["macro_f1"], 3))

## Step 5. 모델 비교

In [ ]:
# TODO: 세 모델의 Accuracy와 Macro F1-score를 표로 비교하세요.
comparison = pd.DataFrame([
    {"model": r["model"], "accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
    for r in [result_a, result_b, result_c]
]).set_index("model")
display(comparison.round(3))
comparison.plot(kind="bar", ylim=(0, 1), figsize=(9, 4))
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: 가장 성능이 높은 모델의 Confusion Matrix를 확인하세요.
best_result = max([result_a, result_b, result_c], key=lambda r: r["macro_f1"])
cm = confusion_matrix(best_result["y_test"], best_result["pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Low","Medium","High"], yticklabels=["Low","Medium","High"])
plt.title(best_result["model"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
print(classification_report(best_result["y_test"], best_result["pred"], target_names=["Low","Medium","High"]))

## Step 6. 변수 중요도와 해석

In [ ]:
# TODO: Model C에서 중요도가 높은 변수를 확인하세요.
importance = pd.DataFrame({
    "feature": result_c["features"],
    "importance": result_c["model_object"].feature_importances_,
}).sort_values("importance", ascending=False)
display(importance.head(15))
importance.head(15).sort_values("importance").plot(kind="barh", x="feature", y="importance", figsize=(8, 6), legend=False)
plt.tight_layout()
plt.show()

## 최종 정리 — 직접 작성

1. 가장 높은 성능을 보인 모델은 무엇인가요?
2. 모델별 Accuracy와 Macro F1-score는 어떻게 달랐나요?
3. 가장 중요한 변수는 무엇이었나요?
4. 해당 변수가 지반함몰 위험과 관련될 수 있는 이유는 무엇인가요?
5. 변수 중요도가 곧 인과관계를 의미하지 않는 이유는 무엇인가요?